# 06. Bounded history enumeration and cancellation

**Level:** Advanced

**Prerequisites:** Lesson 050 and familiarity with paths in a directed graph.

**Canonical claim anchors:** `WDP-CH17-P01`, `WDP-CH17-P02`, `WDP-CH17-N01`

## Learning objectives

- enumerate endpoint-constrained paths under an explicit step budget
- multiply exact edge weights and sum bounded histories
- distinguish aggregate cancellation from pointwise vanishing

Work top-to-bottom. Predict each worked result before executing it, then use
the exercise to test whether you can separate the finite result from its scope
boundary.


In [ ]:
from pathlib import Path
import sys

start = Path.cwd().resolve()
root = next((candidate for candidate in (start, *start.parents)
             if (candidate / 'pyproject.toml').is_file()
             and (candidate / 'code' / 'witnessed_difference').is_dir()), None)
if root is None:
    raise RuntimeError('could not locate the self-contained paper package')
sys.path.insert(0, str(root / 'code'))


## Idea: the budget is part of the mathematical input

A cyclic finite graph can have infinitely many finite paths. The executable
surface therefore refuses enumeration unless the caller supplies a
non-negative maximum number of steps. For the admitted bounded family, each
path weight is the product of edge weights and the history sum is the exact
sum over paths with the declared endpoints.

This is a bounded combinatorial calculation, not an unregulated path integral.


## Visual checkpoint

This lesson uses printed exact data rather than a diagram. Read each returned
status and refusal as part of the result; do not reduce the cell to its final
number.


## Derivation

Starting with the length-zero path `(start,)`, extend the frontier one edge per
round. Record a path whenever its current endpoint equals `end`, while still
allowing extensions until the budget is reached. Sorting first by length and
then lexicographically makes enumeration deterministic. Exact `Fraction`
products prevent rounding-dependent cancellation.


In [ ]:
from fractions import Fraction
from witnessed_difference.histories import FiniteHistoryGraph, WeightedEdge, enumerate_paths, finite_history_sum, path_weight
from witnessed_difference.notebook_support import display_json

graph = FiniteHistoryGraph(
    ('a', 'b', 'c'),
    (
        WeightedEdge('a', 'b', Fraction(1, 2)),
        WeightedEdge('b', 'c', Fraction(2, 3)),
        WeightedEdge('a', 'c', Fraction(-1, 3)),
        WeightedEdge('c', 'a', Fraction(1, 4)),
    ),
)
paths = enumerate_paths(graph, 'a', 'c', 2)
weighted = {"->".join(path): path_weight(graph, path).value for path in paths.value}
print(display_json({'paths': paths, 'weights': weighted, 'sum': finite_history_sum(graph, 'a', 'c', 2)}))


## Read the result

There are two admitted paths with at most two edges: the direct path has weight
`-1/3`, while `a->b->c` has weight `(1/2)(2/3)=1/3`. The aggregate is zero,
but neither path contribution vanishes. That distinction is essential when a
global receipt would otherwise hide local terms.


In [ ]:
from witnessed_difference.histories import cancellation_report

for budget in range(5):
    total = finite_history_sum(graph, 'a', 'c', budget)
    print(display_json({'budget': budget, 'history_sum': total}))
print(display_json(cancellation_report((Fraction(-1, 3), Fraction(1, 3)))))


## Exercise

Why does this cyclic graph require a budget even though the vertex set is
finite? Compute the `a` to `a` sum for budgets 0 and 3; remember that the
length-zero path contributes multiplicative identity `1`.


In [ ]:
answer = {
    'budget_0': finite_history_sum(graph, 'a', 'a', 0),
    'budget_3': finite_history_sum(graph, 'a', 'a', 3),
    'reason': 'the c->a edge permits arbitrarily many cycles as the budget grows',
}
print(display_json(answer))


## Scope check

The finite sum is conditional on a graph, weights, endpoints, and
budget. It is not a continuum path integral or convergence theorem.
**AuthorityEffect: None.**


## Continue the learning path

- Previous: [Witnessed duration and finite multiplicity](050_duration_and_finite_probability.ipynb)
- Index: [Notebook curriculum](README.md)
- Next: [Declared U(1) phases, holonomy, and gauge change](070_declared_u1_holonomy.ipynb)
